In [31]:
import os
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path
from sentence_transformers import SentenceTransformer
import numpy as np
from typing import List,Dict,Set,Any
from chromadb import chromadb
import uuid

In [32]:
def document_load(document_path):
    all_documents=[]
    pdf_dir=Path(document_path)
    pdf_files=list(pdf_dir.glob('**/*.pdf'))
    for pdf_file in pdf_files:
        print(f"Obtained {len(pdf_files)} PDF files")
        try:
            print(f"Reading {pdf_file.name} from the PDFs")
            loader=PyPDFLoader(str(pdf_file))
            documents=loader.load()

            for document in documents:
                document.metadata['source_file']=pdf_file.name
                document.metadata['file_type']='PDF'
            all_documents.extend(documents)
            print(f"Documents loaded with {len(all_documents)} pages")
        except Exception as e:
            print(f"{e}")
    print(f"Document total length {len(all_documents)}")
    return all_documents
        

In [33]:
all_documents_loaded=document_load('../data/')

Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 12 0 (offset 0)
Ignoring wrong pointing object 15 0 (offset 0)
Ignoring wrong pointing object 24 0 (offset 0)
Ignoring wrong pointing object 26 0 (offset 0)


Obtained 1 PDF files
Reading meidtations.pdf from the PDFs
Documents loaded with 128 pages
Document total length 128


In [34]:
def text_splitter(document,chunk_size=1000,chunk_overlap=200):
    textsplitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n","\n"," ",""]
    )   
    split_docs=textsplitter.split_documents(document)
    print(f"Split {len(document)} into {len(split_docs)} chunks")
    return split_docs

chunks=text_splitter(all_documents_loaded)

Split 128 into 576 chunks


In [35]:
# Embedding Class
class Embedding:
    """Converts the document to vector embeddings"""
    def __init__(self,model_name="all-MiniLM-L6-v2"):
        self.model_name=model_name
        self.model=None
        self._load_model()
    def _load_model(self):
        try:
            print(f"Loading {self.model_name} Model")
            self.model=SentenceTransformer(self.model_name)
            print(f"Model loaded succefully with dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model}: {e}")
            raise
    
    def generate_embedding(self,text: List[str])-> np.ndarray:
        if not self.model:
            raise ("Model not loaded")
        print(f"Generating embeddings")
        embedding=self.model.encode(text,show_progress_bar=True)
        print(f"Generated embedding of dimension: {embedding.shape}")
        return embedding


embed=Embedding()
embed         

Loading all-MiniLM-L6-v2 Model


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14966.16it/s]


Model loaded succefully with dimension: 384


In [64]:
# Vector DB
class VectorStore:

    def __init__(self,collection_name="pdf_docs",persistant_directory="../data/vector_store"):
        self.collection_name=collection_name
        self.client=None
        self.collection=None
        self.persistant_directory=persistant_directory
        self._initialize_store()

    def _initialize_store(self):
        try:
            os.makedirs(self.persistant_directory, exist_ok=True)
            self.client=chromadb.PersistentClient(path=self.persistant_directory)

            self.collection=self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description":"Vector store for the RAG"}
            )
        except Exception as e:
            print(f"Error setting up collection: {self.collection_name}")
            raise
    def add_document(self,documents: List[Any], embeddings: np.ndarray):
        if len(documents) != len(embeddings):
            raise (f"The lengths should match for documents and embeddings")

        ids=[]
        metadatas=[]
        document_text=[]
        embedding_list=[]

        for i,(document,embedding) in enumerate(zip(documents,embeddings)):
            doc_id=f"doc_{uuid.uuid4().hex[:8]}_{[i]}"
            ids.append(doc_id)

            metadata=dict(document.metadata)
            metadatas.append(metadata)
            document_text.append(document.page_content)
            embedding_list.append(embedding.tolist())
        
        try:
            self.collection.add(
            ids=ids,
            embeddings=embedding_list,
            metadatas=metadatas,
            documents=document_text
        )
        except Exception as e:
            print(f"Cant add documents")
            raise
vectors=VectorStore()





In [65]:
text=[i.page_content for i in chunks]
embeddings=embed.generate_embedding(text)
vectors.add_document(chunks,embeddings)

Generating embeddings


Batches: 100%|██████████| 18/18 [00:01<00:00, 13.82it/s]


Generated embedding of dimension: (576, 384)


In [67]:
class RagRetriver:
    def __init__(self,vectorstore: VectorStore,embeddingmanager: Embedding):
        self.vectorstore=vectorstore
        self.embeddingmanager=embeddingmanager

    def ragretriever(self, query: str, top_k: int=5, score_threshold: float=0.0)->List[Dict[str,any]]:

        print(f"The query is: {query}")
        print(f"top-k: {top_k}, Score Threshold: {score_threshold}")

        query_embedding=self.embeddingmanager.generate_embedding([query])

        try:
            result=self.vectorstore.collection.query(
                query_embeddings=query_embedding,
                n_results=top_k
            )
            retrieved=[]
            if result['documents'] and result['documents'][0]:
                documents=result['documents'][0]
                metadatas=result['metadatas'][0]
                ids=result['ids'][0]
                distances=result['distances'][0]

                for i,(doc_id,document,metadata,distance) in enumerate(zip(ids,documents,metadatas,distances)):
                    score=1-distance
                    if score>=score_threshold:
                        retrieved.append({
                            'id':doc_id,
                            'content':document,
                            'metadata':metadata,
                            'score':score
                        })
                print(f"Retrieved {len(retrieved)} documents")
            return retrieved
        except Exception as e:
            print(f"Error retrieving")
            raise


retrive=RagRetriver(vectorstore,embeddingmanager)
retrive.ragretriever("meditation")
       


The query is: meditation
top-k: 5, Score Threshold: 0.0
Generating embeddings


Batches: 100%|██████████| 1/1 [00:00<00:00, 40.33it/s]

Generated embedding of dimension: (1, 384)
Retrieved 2 documents


[{'id': 'doc_8e74dead_[348]',
  'content': "MEDITATIONS OF MARCUS AURELIUS \nEIGHTH BOOK \nMarcus Aurelius' Meditations - tr. Casaubon v. 8.16, uploaded to www.philaletheians.co.uk, 14 July 2013 \nPage 76 of 128 \nFor as for that which doth not, it is its own fault and loss, if it bereave itself of her \nlight. \nLV. He that feareth death, either feareth that he shall have no sense at all, or that his \nsenses will not be the same. Whereas, he should rather comfort himself, that either \nno sense at all, and so no sense of evil; or if any sense, then another life, and so no \ndeath properly. \nLVI. All men are made one for another: either then teach them better, or bear with \nthem. \nLVII. The motion of the mind is not as the motion of a dart. For the mind when it is \nwary and cautelous, and by way of diligent circumspection turneth herself many \nways, may then as well be said to go straight on to the object, as when it useth no \nsuch circumspection. \nLVIII. To pierce and penetrat